<a href="https://colab.research.google.com/github/Berkaycetinn/log-analyzer/blob/main/04_kumeleme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q drain3 sentence-transformers
!git clone --depth 1 https://github.com/logpai/loghub.git

import re, json
import numpy as np
import pandas as pd
import torch
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from drain3.masking import MaskingInstruction
from sentence_transformers import SentenceTransformer

pattern = re.compile(
    r"^(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2},\d{3}) "
    r"(?P<level>[A-Z]+) \[(?P<thread>[^\]]*)\] (?P<component>\S+): (?P<message>.*)$"
)
rows = []
with open("loghub/Hadoop/Hadoop_2k.log", encoding="utf-8") as f:
    for line in f:
        m = pattern.match(line.rstrip("\n"))
        if m:
            rows.append(m.groupdict())
df = pd.DataFrame(rows)
df["ts"] = pd.to_datetime(df["date"] + " " + df["time"].str.replace(",", ".", regex=False),
                          format="%Y-%m-%d %H:%M:%S.%f")

MASKS = [
    ("ID",  r"\b(?:application|appattempt|container|job|task|attempt)_[\w]+"),
    ("BLK", r"blk_-?\d+(?:_\d+)?"),
    ("IP",  r"\d+\.\d+\.\d+\.\d+"),
    ("DUR", r"-?\d+(?:ms|s)\b"),
    ("NUM", r"\b\d+\b"),
]
config = TemplateMinerConfig()
config.drain_sim_th = 0.4
config.drain_depth = 4
config.masking_instructions = [MaskingInstruction(rx, name) for name, rx in MASKS]
miner = TemplateMiner(config=config)
df["cluster_id"] = [miner.add_log_message(m)["cluster_id"] for m in df["message"]]
tmap = {c.cluster_id: c.get_template() for c in miner.drain.clusters}
df["template"] = df["cluster_id"].map(tmap)

SUSP = re.compile(r"fail|exception|error|timeout|timed out|refused|no route|unreachable|killed|denied|retry|retrying", re.I)
BENIGN = re.compile(r"maxTaskFailuresPerNode")
df["suspicious"] = df["message"].apply(lambda m: bool(SUSP.search(m)) and not BENIGN.search(m))
df["important"] = df["level"].isin(["ERROR", "WARN", "FATAL"]) | df["suspicious"]

tpl = (df.groupby("cluster_id")
         .agg(template=("template", "first"),
              component=("component", lambda s: s.mode()[0]),
              count=("message", "size"),
              n_important=("important", "sum"),
              first=("ts", "min"), last=("ts", "max"))
         .reset_index())
tpl["important"] = (tpl["n_important"] / tpl["count"]) >= 0.5

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-mpnet-base-v2", device=device)
emb = model.encode(tpl["template"].tolist(), normalize_embeddings=True, show_progress_bar=False)

print("Satır:", len(df), "| Template:", len(tpl), "| Önemli template:", int(tpl["important"].sum()),
      "| Önemli satır:", int(df["important"].sum()), "| Vektör:", emb.shape)
print("Örnek aralığı:", df["ts"].min(), "->", df["ts"].max())

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.12.0 requires cachetools>=5.5, but you have cachetools 4.2.1 which is incompatible.
Cloning into 'loghub'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 91 (delta 8), reused 36 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.09 MiB | 9.11 MiB/s, done.
Resolving deltas: 100% (8/8), done.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Satır: 2000 | Template: 99 | Önemli template: 17 | Önemli satır: 1121 | Vektör: (99, 768)
Örnek aralığı: 2015-10-18 18:01:47.978000 -> 2015-10-18 18:10:55.202000


In [2]:
def extract_params(msg):
    found = {}
    for name, rx in MASKS:
        def repl(m, name=name):
            found.setdefault(name, []).append(m.group(0))
            return f"<{name}>"
        msg = re.sub(rx, repl, msg)
    return found

df["params"] = df["message"].apply(extract_params)

def _num(v):
    m = re.match(r"-?\d+", v)
    return int(m.group()) if m else 0

def field_stats(sub):
    keys = {}
    for p in sub["params"]:
        for name, vals in p.items():
            for k, v in enumerate(vals):
                keys.setdefault(f"{name}#{k+1}", []).append(v)
    out = {}
    for key, vals in keys.items():
        uniq = set(vals)
        if len(uniq) == 1:
            continue
        numeric = key.split("#")[0] in ("NUM", "DUR")
        ordered = sorted(uniq, key=_num) if numeric else sorted(uniq)
        info = {"unique": len(uniq), "example": ordered[:3]}
        if numeric:
            nums = [_num(v) for v in vals]
            info["min"], info["max"] = min(nums), max(nums)
        out[key] = info
    return out

lease_cid = tpl.loc[tpl["template"].str.contains("Failed to renew lease", regex=False), "cluster_id"].iloc[0]
print(field_stats(df[df["cluster_id"] == lease_cid]))

{'NUM#1': {'unique': 326, 'example': ['30', '31', '32'], 'min': 30, 'max': 356}}


In [3]:
from sklearn.cluster import HDBSCAN

def run_hdb(mcs, ms=1):
    return HDBSCAN(min_cluster_size=mcs, min_samples=ms, metric="euclidean").fit_predict(emb)

imp_mask = tpl["important"].values
for mcs in [2, 3, 4, 5, 8]:
    lab = run_hdb(mcs)
    cl = sorted(set(lab) - {-1})
    mixed = sum(1 for c in cl if 0 < imp_mask[lab == c].mean() < 1)
    print(f"min_cluster_size={mcs}  küme={len(cl):2d}  gürültü={int((lab==-1).sum()):2d}  "
          f"önemli+gürültü={int((imp_mask & (lab==-1)).sum()):2d}/{imp_mask.sum()}  "
          f"karışık küme={mixed}")

min_cluster_size=2  küme=25  gürültü=30  önemli+gürültü= 3/17  karışık küme=5
min_cluster_size=3  küme=12  gürültü=44  önemli+gürültü= 4/17  karışık küme=4
min_cluster_size=4  küme= 6  gürültü=34  önemli+gürültü= 1/17  karışık küme=3
min_cluster_size=5  küme= 4  gürültü=42  önemli+gürültü= 1/17  karışık küme=3
min_cluster_size=8  küme= 2  gürültü=42  önemli+gürültü= 6/17  karışık küme=2


In [4]:
MCS = 2
tpl["hdb"] = run_hdb(MCS)

for c in sorted(set(tpl["hdb"]) - {-1}, key=lambda c: -tpl.loc[tpl["hdb"] == c, "count"].sum()):
    g = tpl[tpl["hdb"] == c]
    print(f"\n--- Küme {c}: {len(g)} template, {g['count'].sum()} satır ---")
    for _, r in g.sort_values("count", ascending=False).head(4).iterrows():
        print(f"  {'ÖNEMLİ' if r['important'] else '      '} ({r['count']:3d}x) {r['template'][:90]}")

noise = tpl[tpl["hdb"] == -1]
print(f"\n=== GÜRÜLTÜ: {len(noise)} template, {noise['count'].sum()} satır ===")
print("Bunların önemli olanları (atılmayacak, 'nadir olaylar' bölümüne gidecek):")
for _, r in noise[noise["important"]].sort_values("count", ascending=False).iterrows():
    print(f"  ({r['count']:3d}x) {r['template'][:100]}")


--- Küme 11: 5 template, 626 satır ---
  ÖNEMLİ (476x) Address change detected. Old: msra-sa-<NUM>/<IP>:<NUM> New: msra-sa-<NUM>:<NUM>
  ÖNEMLİ (146x) Retrying connect to server: msra-sa-<NUM>:<NUM>. Already tried <NUM> time(s); retry policy
  ÖNEMLİ (  2x) <NUM> failures on node MININT-FNANLI5.fareast.corp.microsoft.com
         (  1x) Instantiated MRClientService at MININT-FNANLI5.fareast.corp.microsoft.com/<IP>:<NUM>

--- Küme 3: 5 template, 330 satır ---
  ÖNEMLİ (326x) Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for <NUM> seconds. Will re
  ÖNEMLİ (  1x) Slow ReadProcessor read fields took <DUR> (threshold=<DUR>); ack: seqno: -<NUM> status: SU
  ÖNEMLİ (  1x) DFSOutputStream ResponseProcessor exception for block BP-<NUM>-<IP>-<NUM>:<BLK>
  ÖNEMLİ (  1x) Error Recovery for block BP-<NUM>-<IP>-<NUM>:<BLK> in pipeline <IP>:<NUM>, <IP>:<NUM>: bad

--- Küme 23: 2 template, 290 satır ---
         (289x) Progress of TaskAttempt <ID> is : <NUM>.<NUM>
         (  1x) N

In [5]:
imp = df[df["important"]].sort_values("ts").copy()

print("GAP duyarlılığı (kaç olay penceresi çıkıyor):")
for gap in [5, 10, 30, 60, 120]:
    n = (imp["ts"].diff() > pd.Timedelta(seconds=gap)).cumsum().nunique()
    print(f"  GAP={gap:3d} sn -> {n} pencere")

GAP = 30       # saniye
imp["win"] = (imp["ts"].diff() > pd.Timedelta(seconds=GAP)).cumsum()

win = imp.groupby("win").agg(başlangıç=("ts", "min"), bitiş=("ts", "max"),
                             satır=("message", "size"), template=("cluster_id", "nunique"))
win["süre_sn"] = (win["bitiş"] - win["başlangıç"]).dt.total_seconds().round(1)
print()
print(win.to_string())

GAP duyarlılığı (kaç olay penceresi çıkıyor):
  GAP=  5 sn -> 3 pencere
  GAP= 10 sn -> 3 pencere
  GAP= 30 sn -> 3 pencere
  GAP= 60 sn -> 1 pencere
  GAP=120 sn -> 1 pencere

                  başlangıç                   bitiş  satır  template  süre_sn
win                                                                          
0   2015-10-18 18:04:11.034 2015-10-18 18:04:11.034      1         1      0.0
1   2015-10-18 18:04:52.552 2015-10-18 18:04:52.552      1         1      0.0
2   2015-10-18 18:05:27.570 2015-10-18 18:10:55.202   1119        16    327.6


In [6]:
cid2hdb = tpl.set_index("cluster_id")["hdb"]
imp["hdb"] = imp["cluster_id"].map(cid2hdb)

for w, g in imp.groupby("win"):
    print(f"\n=== Pencere {w}: {g['ts'].min().time()} -> {g['ts'].max().time()} | "
          f"{len(g)} satır | {g['cluster_id'].nunique()} template | "
          f"{g['hdb'].nunique()} farklı embedding kümesi ===")
    for cid, gg in sorted(g.groupby("cluster_id"), key=lambda x: -len(x[1])):
        print(f"  küme {int(gg['hdb'].iloc[0]):3d} ({len(gg):3d}x) {tmap[cid][:85]}")


=== Pencere 0: 18:04:11.034000 -> 18:04:11.034000 | 1 satır | 1 template | 1 farklı embedding kümesi ===
  küme  21 (  1x) Container complete event for unknown container id <ID>

=== Pencere 1: 18:04:52.552000 -> 18:04:52.552000 | 1 satır | 1 template | 1 farklı embedding kümesi ===
  küme  -1 (  1x) Diagnostics report from <ID>: Container killed by the ApplicationMaster.

=== Pencere 2: 18:05:27.570000 -> 18:10:55.202000 | 1119 satır | 16 template | 8 farklı embedding kümesi ===
  küme  11 (476x) Address change detected. Old: msra-sa-<NUM>/<IP>:<NUM> New: msra-sa-<NUM>:<NUM>
  küme   3 (326x) Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for <NUM> seconds. Wi
  küme  -1 (147x) ERROR IN CONTACTING RM.
  küme  11 (146x) Retrying connect to server: msra-sa-<NUM>:<NUM>. Already tried <NUM> time(s); retry p
  küme   6 (  6x) <ID> <*> Transitioned from <*> to <*>
  küme  15 (  4x) Diagnostics report from <ID>: Error: java.net.NoRouteToHostException: No Route to Hos
  küme

In [7]:
GAP = 30
CONTEXT_SEC = 120

imp = df[df["important"]].sort_values("ts").copy()
imp["win"] = (imp["ts"].diff() > pd.Timedelta(seconds=GAP)).cumsum()

def template_item(gg, cid):
    return {
        "template": tmap[cid],
        "example": gg["message"].iloc[0][:250],
        "count": int(len(gg)),
        "first": str(gg["ts"].min().time()),
        "last": str(gg["ts"].max().time()),
        "levels": {k: int(v) for k, v in gg["level"].value_counts().items()},
        "varying_fields": field_stats(gg),
        "embedding_cluster": int(cid2hdb[cid]),
    }

events = []
for w, g in imp.groupby("win"):
    start = g["ts"].min()
    items = sorted((template_item(gg, cid) for cid, gg in g.groupby("cluster_id")),
                   key=lambda x: -x["count"])
    before = imp[(imp["ts"] < start) & (imp["ts"] >= start - pd.Timedelta(seconds=CONTEXT_SEC))]
    events.append({
        "event_id": int(w),
        "start": str(start),
        "end": str(g["ts"].max()),
        "n_rows": int(len(g)),
        "templates": items,
        "context_before": [{"time": str(r.ts.time()), "level": r.level, "message": r.message[:200]}
                           for r in before.itertuples()],
    })

out = {
    "sample_start": str(df["ts"].min()),
    "sample_end": str(df["ts"].max()),
    "n_rows_total": int(len(df)),
    "note": "Log örneği sample_end anında kesiliyor; o ana kadar bitmemiş bir olayın çözüldüğü varsayılamaz.",
    "events": events,
}
with open("events.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

txt = json.dumps(out, ensure_ascii=False)
print("JSON boyutu:", len(txt), "karakter (yaklaşık", len(txt) // 4, "token)")
for e in events:
    print(f"olay {e['event_id']}: {e['start'][11:23]} -> {e['end'][11:23]} | {e['n_rows']} satır | "
          f"{len(e['templates'])} template | {len(e['context_before'])} önceki olay")

big = max(events, key=lambda e: e["n_rows"])
for t in big["templates"][:4]:
    print(f"\n{t['count']}x {t['template'][:70]}")
    print("  örnek:", t["example"][:130])
    print("  değişen alanlar:", json.dumps(t["varying_fields"], ensure_ascii=False)[:160])

JSON boyutu: 8973 karakter (yaklaşık 2243 token)
olay 0: 18:04:11.034 -> 18:04:11.034 | 1 satır | 1 template | 0 önceki olay
olay 1: 18:04:52.552 -> 18:04:52.552 | 1 satır | 1 template | 1 önceki olay
olay 2: 18:05:27.570 -> 18:10:55.202 | 1119 satır | 16 template | 2 önceki olay

476x Address change detected. Old: msra-sa-<NUM>/<IP>:<NUM> New: msra-sa-<N
  örnek: Address change detected. Old: msra-sa-41/10.190.173.170:9000 New: msra-sa-41:9000
  değişen alanlar: {"NUM#2": {"unique": 2, "example": ["8030", "9000"], "min": 8030, "max": 9000}, "NUM#4": {"unique": 2, "example": ["8030", "9000"], "min": 8030, "max": 9000}}

326x Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for <N
  örnek: Failed to renew lease for [DFSClient_NONMAPREDUCE_1537864556_1] for 30 seconds.  Will retry shortly ...
  değişen alanlar: {"NUM#1": {"unique": 326, "example": ["30", "31", "32"], "min": 30, "max": 356}}

147x ERROR IN CONTACTING RM.
  örnek: ERROR IN CONTACTING RM. 
  değişen alanlar: 

In [8]:
pd.set_option("display.max_colwidth", 140)

def port_table(key, pat):
    s = df[df["message"].str.contains(key, regex=False)].copy()
    s["port"] = s["message"].str.extract(pat)[0]
    print(key)
    print(s.groupby("port").agg(satır=("ts", "size"), ilk=("ts", "min"), son=("ts", "max")).to_string(), "\n")

port_table("Address change detected", r"New: \S+:(\d+)")
port_table("Retrying connect", r"msra-sa-\d+:(\d+)")

r = df[df["message"].str.contains("Retrying connect", regex=False)]["message"]
print(r.iloc[0]); print(r.iloc[-1])

for key in ["KILLING", "Cannot assign container"]:
    s = df[df["message"].str.contains(key, regex=False)]
    print(f"\n{key}: {len(s)} satır, seviye: {s['level'].unique().tolist()}")
    print(s[["time", "message"]].head(3).to_string(index=False))

Address change detected
      satır                     ilk                     son
port                                                       
8030    146 2015-10-18 18:06:02.841 2015-10-18 18:10:53.546
9000    330 2015-10-18 18:05:27.570 2015-10-18 18:10:55.202 

Retrying connect
      satır                     ilk                     son
port                                                       
8030    146 2015-10-18 18:06:03.856 2015-10-18 18:10:54.546 

Retrying connect to server: msra-sa-41:8030. Already tried 0 time(s); retry policy is RetryUpToMaximumCountWithFixedSleep(maxRetries=10, sleepTime=1000 MILLISECONDS)
Retrying connect to server: msra-sa-41:8030. Already tried 0 time(s); retry policy is RetryUpToMaximumCountWithFixedSleep(maxRetries=10, sleepTime=1000 MILLISECONDS)

KILLING: 3 satır, seviye: ['INFO']
        time                                       message
18:04:50,208 KILLING attempt_1445144423722_0020_m_000003_0
18:06:26,029 KILLING attempt_1445144423722_0020_m

In [9]:
for key in ["getResources()", "Progress of TaskAttempt"]:
    s = df[df["message"].str.contains(key, regex=False)]
    print(f"{key}: {len(s)} satır | {s['ts'].min().time()} -> {s['ts'].max().time()}")

per_min = df.groupby([df["ts"].dt.strftime("%H:%M"), "important"]).size().unstack(fill_value=0)
print(per_min.to_string())

all_rows = df.sort_values("ts")
for e in events:
    start = pd.Timestamp(e["start"])
    prev = all_rows[all_rows["ts"] < start].tail(5)
    e["preceding_lines"] = [{"time": str(r.ts.time())[:12], "level": r.level, "message": r.message[:200]}
                            for r in prev.itertuples()]
out["events"] = events
with open("events.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print("\nOlay 0 öncesi 5 satır:")
for l in events[0]["preceding_lines"]:
    print(" ", l["time"], l["level"], l["message"][:110])

getResources(): 12 satır | 18:01:54.838000 -> 18:04:52.552000
Progress of TaskAttempt: 289 satır | 18:02:17.027000 -> 18:06:28.170000
important  False  True 
ts                     
18:01        157      0
18:02        188      0
18:03        232      0
18:04        266      2
18:05          2     71
18:06         34    226
18:07          0    210
18:08          0    210
18:09          0    210
18:10          0    192

Olay 0 öncesi 5 satır:
  18:04:10.002 INFO Reduce slow start threshold not met. completedMapsForReduceSlowstart 1
  18:04:10.002 INFO After Scheduling: PendingReds:1 ScheduledMaps:0 ScheduledReds:0 AssignedMaps:10 AssignedReds:0 CompletedMaps:0
  18:04:10.315 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000002_0 is : 0.27772525
  18:04:10.424 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000000_0 is : 0.27696857
  18:04:10.940 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000005_0 is : 0.10685723


In [10]:
t0 = pd.Timestamp("2015-10-18 18:05:27.570")
pre = df[(df["ts"] < t0) & (~df["important"])].tail(8)
print("Fırtına öncesi son normal satırlar:")
for r in pre.itertuples():
    print(" ", r.time, r.level, r.message[:105])

prog = df[df["message"].str.contains("Progress of TaskAttempt", regex=False)].copy()
prog["att"] = prog["message"].str.extract(r"(attempt_\S+)")[0]
print("\nAttempt bazında Progress raporları:")
print(prog.groupby("att")["ts"].agg(satır="count", ilk="min", son="max").to_string())

act = df.groupby([df["ts"].dt.strftime("%H:%M"), "important"]).size().unstack(fill_value=0)
out["activity_per_minute"] = {k: {"normal": int(r.get(False, 0)), "flagged": int(r.get(True, 0))}
                              for k, r in act.iterrows()}

def related(e, window=120, cap=15):
    ids = set()
    for t in e["templates"]:
        ids.update(re.findall(r"(?:container|attempt|job|task)_[\w]+", t["example"]))
    if not ids:
        return []
    lo = pd.Timestamp(e["start"]) - pd.Timedelta(seconds=window)
    hi = pd.Timestamp(e["end"]) + pd.Timedelta(seconds=window)
    sub = df[(df["ts"] >= lo) & (df["ts"] <= hi)]
    sub = sub[~sub["message"].str.contains("Progress of TaskAttempt", regex=False)]
    hit = sub[sub["message"].apply(lambda m: any(i in m for i in ids))]
    return [{"time": str(r.ts.time())[:12], "level": r.level, "message": r.message[:220]}
            for r in hit.head(cap).itertuples()]

for e in events:
    e["related_lines"] = related(e) if e["n_rows"] <= 5 else []
out["events"] = events

with open("events.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print("\nJSON boyutu:", len(json.dumps(out, ensure_ascii=False)) // 4, "token (yaklaşık)")
for e in events[:2]:
    print(f"\nOlay {e['event_id']} ilişkili satırlar ({len(e['related_lines'])}):")
    for l in e["related_lines"]:
        print(" ", l["time"], l["level"], l["message"][:120])

Fırtına öncesi son normal satırlar:
  18:04:56,568 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000000_0 is : 0.3624012
  18:04:57,396 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000009_0 is : 0.76133776
  18:04:57,427 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000008_0 is : 0.34610128
  18:04:57,443 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000007_0 is : 0.3707891
  18:04:59,771 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000001_0 is : 0.37551183
  18:04:59,787 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000002_0 is : 0.38137424
  18:05:02,802 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000001_0 is : 0.37551183
  18:05:02,818 INFO Progress of TaskAttempt attempt_1445144423722_0020_m_000002_0 is : 0.38137424

Attempt bazında Progress raporları:
                                       satır                     ilk                     son
att                                                 

In [11]:
rows_out = []
for a in sorted(prog["att"].unique()):
    p = prog[prog["att"] == a]
    last = p.iloc[-1]
    val = re.search(r"is : ([\d.]+)", last["message"]).group(1)
    other = df[df["message"].str.contains(a, regex=False)
               & ~df["message"].str.contains("Progress of TaskAttempt", regex=False)
               & (df["ts"] > last["ts"])]
    rows_out.append({"attempt": a[-10:], "son_rapor": str(last["ts"].time())[:12], "son_değer": val,
                     "sonrası_satır": len(other),
                     "sonraki_ilk": str(other.iloc[0]["ts"].time())[:12] if len(other) else "-"})
print(pd.DataFrame(rows_out).to_string(index=False))

for a in ["m_000001_0", "m_000002_0"]:
    p = prog[prog["att"].str.endswith(a)].tail(10)
    vals = p["message"].str.extract(r"is : ([\d.]+)")[0]
    print("\n", a)
    print(pd.DataFrame({"zaman": p["ts"].dt.strftime("%H:%M:%S.%f").str[:12], "değer": vals}).to_string(index=False))

m6 = df[(df["ts"] >= "2015-10-18 18:06:00") & (df["ts"] < "2015-10-18 18:07:00") & (~df["important"])]
print("\n18:06 normal satır:", len(m6))
print(m6.assign(t=m6["template"].str[:80]).groupby("t")["ts"].agg(satır="size", ilk="min", son="max").to_string())

   attempt    son_rapor  son_değer  sonrası_satır  sonraki_ilk
m_000000_0 18:04:56.568  0.3624012              0            -
m_000001_0 18:06:28.170 0.37551183              9 18:06:28.217
m_000002_0 18:06:25.998 0.38137424              9 18:06:26.029
m_000003_0 18:04:49.895        1.0              7 18:04:50.208
m_000004_0 18:04:54.708 0.44968578              0            -
m_000005_0 18:04:54.693 0.44950968              0            -
m_000006_0 18:04:55.630 0.44980705              0            -
m_000007_0 18:04:57.443  0.3707891              0            -
m_000008_0 18:04:57.427 0.34610128              0            -
m_000009_0 18:04:57.396 0.76133776              0            -

 m_000001_0
       zaman      değer
18:05:02.802 0.37551183
18:06:06.169 0.37551183
18:06:09.325 0.37551183
18:06:12.466 0.37551183
18:06:15.623 0.37551183
18:06:18.763 0.37551183
18:06:21.904 0.37551183
18:06:25.092 0.37551183
18:06:28.123 0.37551183
18:06:28.170 0.37551183

 m_000002_0
       zaman     

In [12]:
for a in ["m_000001_0", "m_000002_0"]:
    p = prog[prog["att"].str.endswith(a) & (prog["ts"] >= "2015-10-18 18:04:20") & (prog["ts"] <= "2015-10-18 18:05:03")]
    vals = p["message"].str.extract(r"is : ([\d.]+)")[0]
    print("\n", a)
    print(pd.DataFrame({"zaman": p["ts"].dt.strftime("%H:%M:%S.%f").str[:12], "değer": vals}).to_string(index=False))

att_rows = []
for a in sorted(prog["att"].unique()):
    p = prog[prog["att"] == a]
    vals = p["message"].str.extract(r"is : ([\d.]+)")[0]
    same_tail = int((vals == vals.iloc[-1]).iloc[::-1].astype(int).cumprod().sum())
    gp = p["ts"].diff().fillna(pd.Timedelta(0))
    pos = int(gp.values.argmax())
    after = df[df["message"].str.contains(a, regex=False)
               & ~df["message"].str.contains("Progress of TaskAttempt", regex=False)
               & (df["ts"] > p["ts"].iloc[-1])]
    if after["message"].str.contains("to FAILED", regex=False).any():
        outcome = "FAILED"
    elif after["message"].str.contains("to SUCCEEDED", regex=False).any():
        outcome = "SUCCEEDED"
    else:
        outcome = "no outcome in sample"
    att_rows.append({
        "attempt": a[-10:],
        "first_report": str(p["ts"].iloc[0].time())[:12],
        "last_report": str(p["ts"].iloc[-1].time())[:12],
        "last_value": vals.iloc[-1],
        "identical_tail_reports": same_tail,
        "longest_report_gap_sec": round(gp.iloc[pos].total_seconds(), 1),
        "gap_from": str(p["ts"].iloc[pos - 1].time())[:12] if pos > 0 else None,
        "gap_to": str(p["ts"].iloc[pos].time())[:12] if pos > 0 else None,
        "outcome": outcome,
    })
print("\n", pd.DataFrame(att_rows).to_string(index=False))

allts = df["ts"].sort_values().reset_index(drop=True)
g = allts.diff().fillna(pd.Timedelta(0))
pos = int(g.values.argmax())
silence = {"from": str(allts.iloc[pos - 1].time())[:12], "to": str(allts.iloc[pos].time())[:12],
           "seconds": round(g.iloc[pos].total_seconds(), 1)}
print("\nEn uzun sessizlik (tüm satırlar):", silence)

big = max(events, key=lambda e: e["n_rows"])
t0 = pd.Timestamp(big["start"])
first_normal = df[(df["ts"] >= t0) & (~df["important"])]["ts"].min()
print("Fırtına başlangıcı:", big["start"][11:23], "| fırtınadan sonraki ilk normal satır:", str(first_normal.time())[:12])

out["pre_storm_facts"] = {
    "note": "Yorumsuz olgular: log satırlarından hesaplandı, sebep iddiası içermez.",
    "map_attempt_progress": att_rows,
    "longest_silence_all_lines": silence,
    "largest_event_first_line": big["start"][11:23],
    "first_normal_line_after_largest_event_start": str(first_normal.time())[:12],
}
with open("events.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print("JSON boyutu:", len(json.dumps(out, ensure_ascii=False)) // 4, "token (yaklaşık)")


 m_000001_0
       zaman      değer
18:04:20.206  0.3512319
18:04:23.706 0.36319977
18:04:27.160 0.36319977
18:04:30.644 0.36319977
18:04:34.098 0.36319977
18:04:37.754 0.36319977
18:04:41.270 0.36319977
18:04:44.864 0.36319977
18:04:50.208 0.36319977
18:04:53.286 0.37225527
18:04:56.318 0.37322965
18:04:59.771 0.37551183
18:05:02.802 0.37551183

 m_000002_0
       zaman      değer
18:04:20.394 0.36317363
18:04:23.972 0.36317363
18:04:27.269 0.36317363
18:04:30.816 0.36317363
18:04:34.363 0.36317363
18:04:37.957 0.36317363
18:04:41.535 0.36317363
18:04:45.176 0.36317363
18:04:50.286  0.3673702
18:04:53.364  0.3787692
18:04:56.396 0.38007197
18:04:59.787 0.38137424
18:05:02.818 0.38137424

    attempt first_report  last_report last_value  identical_tail_reports  longest_report_gap_sec     gap_from       gap_to              outcome
m_000000_0 18:02:17.027 18:04:56.568  0.3624012                       8                     4.0 18:03:45.345 18:03:49.360 no outcome in sample
m_000001_0 18: